In [ ]:
import warnings
warnings.filterwarnings('ignore')
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "utils")))
from analysis_utils import (
    volcano_plot_br,
    plot_spatial_clusters_per_sample,
    plot_dotplot_by_treatment,
    plot_relative_cluster_composition,
    score_and_plot_modules, 
    volcano_plot_region_within_group
)
import scanpy as sc
import numpy as np

import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import pandas as pd
import scanpy as sc
from itertools import combinations
from scipy import sparse

# ---- 1) Run DE for many contrasts once and cache tidy results ----
def run_de_contrasts(
    adata,
    groupby="treatment",
    contrasts="all_pairwise",     # "all_pairwise" or list of (group, reference)
    layer="counts",
    method="wilcoxon",
    lfc_cap=10.0,                 # cap absurd LFCs for stability
):
    cats = adata.obs[groupby].astype(str)
    levels = list(pd.Categorical(cats).categories) or sorted(cats.unique())
    if contrasts == "all_pairwise":
        contrasts = [(g, r) for g, r in combinations(levels, 2)] + [(r, g) for g, r in combinations(levels, 2)]

    results = []
    for group, reference in contrasts:
        sc.tl.rank_genes_groups(
            adata, groupby=groupby, groups=[group], reference=reference,
            method=method, layer=layer, use_raw=False
        )
        de = sc.get.rank_genes_groups_df(adata, group=group)

        # sanitize/cap LFCs and fallback if missing
        de = de.rename(columns={"names":"gene","pvals":"pval","pvals_adj":"padj","logfoldchanges":"log2fc"})
        de["padj"] = de["padj"].fillna(de["pval"])

        # fallback LFC if NA/inf
        need_fallback = (~np.isfinite(de["log2fc"])).all() or de["log2fc"].isna().all()
        if need_fallback:
            X = adata.layers.get(layer, adata.X)
            if sparse.issparse(X): X = X.toarray()
            mask_g = (adata.obs[groupby].astype(str).values == str(group))
            mask_r = (adata.obs[groupby].astype(str).values == str(reference))
            eps = 1e-9
            mean_g = X[mask_g].mean(axis=0); mean_r = X[mask_r].mean(axis=0)
            lfc_fallback = np.log2((mean_g + eps) / (mean_r + eps))
            de = de.merge(pd.Series(lfc_fallback, index=adata.var_names, name="log2fc_fallback"),
                          left_on="gene", right_index=True, how="left")
            de["log2fc"] = de["log2fc_fallback"].fillna(de["log2fc"])

        # cap extremes
        de["log2fc"] = de["log2fc"].clip(-lfc_cap, lfc_cap)

        de["groupby"]   = groupby
        de["group"]     = group
        de["reference"] = reference
        de["contrast"]  = f"{group}__vs__{reference}"

        # keep tidy columns
        keep_cols = ["contrast","groupby","group","reference","gene","log2fc","pval","padj","scores","pts","pts_rest"]
        for c in keep_cols:
            if c not in de.columns: de[c] = np.nan
        results.append(de[keep_cols])

    de_all = pd.concat(results, ignore_index=True)
    return de_all

# ---- 2) Slice what you need later (genes and/or contrasts) ----
def select_de(
    de_all,
    genes=None,                             # list or None
    contrasts=None,                         # list like ["BRICHOS__vs__PBS", ...] or None
    padj_max=None, lfc_min=None
):
    df = de_all.copy()
    if genes is not None:
        genes = [str(g) for g in genes]
        df = df[df["gene"].isin(genes)]
    if contrasts is not None:
        df = df[df["contrast"].isin(contrasts)]
    if padj_max is not None:
        df = df[df["padj"] <= padj_max]
    if lfc_min is not None:
        df = df[df["log2fc"].abs() >= lfc_min]
    # nice ordering
    return df.sort_values(["contrast","padj","log2fc"], ascending=[True, True, False])

# ---- 3) (Optional) Pivot for a quick matrix view per contrast ----
def pivot_de(de_sub):
    return de_sub.pivot_table(index="gene", columns="contrast", values="log2fc")
    
pig_genes  = [
    "Apoe", "Arpc1b", "Axl", "B2m", "C1qa", "C1qb", "C1qc", "C4b",
    "Cd63", "Cd9", "Clu", "Csf1r", "Cst3", "Ctsa", "Ctsb", "Ctsd",
    "Ctsh", "Ctsl", "Ctss", "Ctsz", "Cx3cr1", "Cyba", "Fcer1g", "Fcgr3",
    "Fcrls", "Gfap", "Gns", "Grn", "Gusb", "H2-D1", "H2-K1", "Hexa",
    "Hexb", "Igfbp5", "Itgb5", "Itm2b", "Laptm5", "Lgals3bp", "Lgmn", "Ly86",
    "Lyz2", "Man2b1", "Mpeg1", "Npc2", "Olfml3", "Plek", "Prdx6",
    "S100a6", "Serpina3n", "Trem2", "Tyrobp", "Vsir"
]
olig_genes = [
    "Plp1", "Mbp", "Mobp", "Cnp", "Cldn11", "Mal", "Apod", "Trf", "Fth1", "Plekhb1", "Ppp1r14a", "Ttyh2", "Fa2h", "Aspa"
]
modules = {
    "PIG": pig_genes,
    "OLIG": olig_genes
}




In [ ]:
ad = sc.read_h5ad('../data/ST_BRICHOS_region.h5ad')

In [ ]:
sc.pl.dotplot(
        ad,
        var_names=['Itm2b','Itm2c'],
        groupby="treatment",
        standard_scale="var",
        color_map="Reds",
        dendrogram=True,
        figsize=(4, 2)
    )

In [ ]:
# 1) Compute once (all pairwise comparisons across your `treatment` groups)
de_all = run_de_contrasts(ad, groupby="treatment", contrasts="all_pairwise",layer = None, method="wilcoxon")

# 2) Later, pull only what you want—e.g., Itm2b/Itm2c in two contrasts
want_genes = ["Itm2b","Itm2c"]
want_contrasts = ["BRICHOS__vs__PBS","PBS__vs__WT"]   # adjust to your labels
de_sub = select_de(de_all, genes=want_genes, contrasts=want_contrasts)

# 3) (Optional) pretty matrix of log2FCs
logfc_mat = pivot_de(de_sub)
print(logfc_mat)

# 4) Filtered significant
sig_sub = select_de(de_all, genes=want_genes, contrasts=want_contrasts, padj_max=0.05)

In [ ]:
sig_sub